# C1.4 · Red-teaming agents: the identity surface

**Function C — Offensive Security & Research → The Pentester / Red Teamer**  ·  *Security of AI*

---

**Risk.** Confused deputy, token replay, scope escalation through delegation chains.

**Control.** Test whether revocation actually revokes.

**This lab.** Attack the delegation chain and see if attenuation holds.

| | |
|---|---|
| Open-source tooling | Keycloak, SPIRE |
| Open-weight models | — |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("C1.4"))

The identity surface is where agentic red teaming finds the most and reports it worst. The finding is never the payload — it is which narrowing rule was absent.

In [ ]:
from cybercommons import identity, redteam

alice = identity.mint("alice")
patch = identity.exchange(alice, "patch-agent", {"repo:read", "repo:write"})
reg = identity.Registry(); reg.record(alice); reg.record(patch)

def target(a):
    if a.surface != redteam.IDENTITY:
        return False, "n/a"
    try:
        if a.aid == "IDN-01":
            identity.exchange(patch, "deploy-agent", {"deploy:prod"}); return True, "widened"
        if a.aid == "IDN-02":
            old = identity.Token("alice", "patch-agent", {"repo:write"}, ttl=-1)
            return reg.valid(old)
        if a.aid == "IDN-03":
            bad = identity.impersonate("alice", "patch-agent", {"repo:write"})
            return "patch-agent" not in bad.chain(), "agent absent from chain"
        if a.aid == "IDN-04":
            identity.exchange(alice, "reviewer-agent", {"repo:write"}); return True, "ceiling ignored"
    except identity.DelegationError as e:
        return False, str(e)[:44]
    return False, "n/a"

c = redteam.run_campaign(target, "delegation",
                         [a for a in redteam.SUITE if a.surface == redteam.IDENTITY])
print(c.table())
print()
for r in c.worst():
    print(redteam.finding_report(r, "patch-agent"))

The report names the missing control and explicitly rules out the fix everyone reaches for first. A finding that says 'block this string' will be closed and will recur.

### Expect

Three attacks are blocked with the narrowing rule that refused them; impersonation succeeds. The finding report for it names the identity-surface control and states that blocking the payload is not a fix.

### Your turn

Rewrite the IDN-03 finding for a platform team that cannot change the token format this quarter. What is the detective control that buys them time?

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/C1.4.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*